# 🧑 Notebook 4: Human Annotation & Metric–Human Misalignment

**Reads:** `metrics_all_models.csv`  
**Writes:** `annotated_all_models.csv`, `misalignment_summary.csv`, `iaa_scores.csv`

---

## ⚡ If you have a real annotation CSV

Set `REAL_ANNOTATION_CSV` in **Cell 3** to your file path.  
Required columns:
```
image_id, model, human_correct (0/1), error_type, error_token
```
If the path doesn't exist, the notebook falls back to simulation.


In [ ]:
# CELL 1 — Install
import subprocess, sys
for pkg in ['pandas','numpy','matplotlib','seaborn','scipy','scikit-learn','tqdm']:
    subprocess.check_call([sys.executable,'-m','pip','install',pkg])
print('✅ Done.')

In [ ]:
# CELL 2 — Imports & config
import sys, warnings, random
from pathlib import Path
from collections import Counter
from typing import List

sys.path.insert(0, str(Path('.')))
import config as cfg

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import (
    cohen_kappa_score, confusion_matrix, classification_report
)
from tqdm import tqdm
warnings.filterwarnings('ignore')

SEED        = cfg.SEED
MODELS      = cfg.MODELS_TO_RUN
ERROR_TYPES = cfg.ERROR_TYPES
OUTPUT_DIR  = cfg.OUTPUT_DIR
FIGURES_DIR = cfg.FIGURES_DIR
random.seed(SEED); np.random.seed(SEED)
print('✅ Config ready.')

In [ ]:
# CELL 3 — ★ REAL ANNOTATION CSV PATH ★
# If you have annotated captions, point this to your CSV.
# Otherwise leave as None and simulation will be used.
#
# Supported CSV formats:
#
# FORMAT A — Wide (one row per image, per-model suffixed columns):
#   image_id, human_correct_blip, human_correct_blip2, human_correct_vit_gpt2,
#   error_type_blip, error_type_blip2, error_type_vit_gpt2,
#   error_token_blip, ..., annotator_1_blip, annotator_2_blip, annotator_3_blip, ...
#   ← This is what captions_all_models.csv produces.
#
# FORMAT B — Long (one row per image-model pair):
#   image_id (int), model (str: blip/blip2/vit_gpt2),
#   human_correct (0/1), error_type (str), error_token (str),
#   annotator_1 (0/1), annotator_2 (0/1), annotator_3 (0/1)
#
# Example:
#   REAL_ANNOTATION_CSV = Path('my_annotations.csv')

REAL_ANNOTATION_CSV = 'E:/DL/minor/code_2/research_data/outputs/captions_all_models (1).csv'

USE_REAL = REAL_ANNOTATION_CSV is not None and Path(REAL_ANNOTATION_CSV).exists()
print(f'Annotation mode: {"REAL FILE" if USE_REAL else "SIMULATION"}')
if USE_REAL:
    print(f'  File: {REAL_ANNOTATION_CSV}')

In [ ]:
# CELL 4 — Load metrics DataFrame
metrics_path = OUTPUT_DIR / 'metrics_all_models.csv'
if not metrics_path.exists():
    raise FileNotFoundError('Run 03_metric_computation.ipynb first.')

df = pd.read_csv(metrics_path)
print(f'✅ Loaded: {len(df)} rows × {len(df.columns)} cols')
df[['image_id','scenario_category','primary_gt_caption']].head(3)

---
## 📝 Section 1: Load or Simulate Annotations

In [ ]:
# CELL 5 — Load REAL annotations if available
#
# FIX: The annotation CSV is in WIDE format (one row per image, columns like
#      human_correct_blip, error_type_blip2, annotator_1_vit_gpt2, etc.).
#
# The previous code assumed LONG format (a 'model' column + one row per image-model),
# which caused the merge to silently produce all-NaN annotation columns.
#
# This cell now handles BOTH formats:
#   - Wide: columns named  {col}_{model}  (e.g. human_correct_blip)
#   - Long: a 'model' column + generic column names  (e.g. human_correct)

if USE_REAL:
    ann = pd.read_csv(REAL_ANNOTATION_CSV)
    print(f'✅ Real annotations loaded: {len(ann)} rows')
    print(f'   Columns: {ann.columns.tolist()}')

    ANN_COLS = ['human_correct', 'error_type', 'error_token',
                'annotator_1', 'annotator_2', 'annotator_3']

    # ── Detect format ──────────────────────────────────────────────────────────
    # Wide format: at least one column of the form  {ann_col}_{model}  exists
    wide_cols = [c for c in ann.columns
                 for m in MODELS
                 for a in ANN_COLS
                 if c == f'{a}_{m}']
    is_wide = len(wide_cols) > 0

    if is_wide:
        # ── WIDE-FORMAT merge ──────────────────────────────────────────────────
        # The annotation file already has per-model suffixed columns.
        # Simply merge on image_id and let pandas align by index.
        ann_indexed = ann.set_index('image_id')
        for model in MODELS:
            for col in ANN_COLS:
                wide_col = f'{col}_{model}'
                if wide_col in ann_indexed.columns:
                    df[wide_col] = df['image_id'].map(ann_indexed[wide_col])
        print('✅ Wide-format annotations merged into df.')

    else:
        # ── LONG-FORMAT merge (original logic, now correct) ────────────────────
        # Annotation file has a 'model' column; one row per (image_id, model).
        if 'model' not in ann.columns:
            raise ValueError(
                "Annotation CSV is not wide-format (no {col}_{model} columns) "
                "and has no 'model' column for long-format. Cannot merge."
            )
        for model in MODELS:
            model_ann = ann[ann['model'] == model].set_index('image_id')
            for col in ANN_COLS:
                if col in model_ann.columns:
                    df[f'{col}_{model}'] = df['image_id'].map(model_ann[col])
        print('✅ Long-format annotations merged into df.')

    # ── Verify merge ───────────────────────────────────────────────────────────
    for model in MODELS:
        hc_col = f'human_correct_{model}'
        if hc_col in df.columns:
            n_filled = df[hc_col].notna().sum()
            print(f'   {model}: {n_filled}/{len(df)} human_correct values filled')

else:
    print('⚠️  No real annotation file found — running simulation.')
    print('   To use real annotations: set REAL_ANNOTATION_CSV in Cell 3.')


In [ ]:
# CELL 6 — Simulation (runs only if real annotations not available)
if not USE_REAL:
    # Error rate by scenario (encodes known difficulty from literature)
    SCENARIO_ERR_RATE = {
        'gender_ambiguity':0.38, 'object_confusion':0.35,
        'context_mismatch':0.30, 'counting_errors':0.28,
        'attribute_errors':0.25, 'general_baseline':0.18
    }
    # Error type distribution per model (calibrated from midterm + literature)
    MODEL_ERR_DIST = {
        'blip':     [0.45, 0.25, 0.18, 0.12],
        'blip2':    [0.38, 0.28, 0.20, 0.14],
        'ofa':      [0.40, 0.30, 0.18, 0.12],
        'vit_gpt2': [0.50, 0.22, 0.16, 0.12],
    }

    for model in MODELS:
        rng  = np.random.RandomState(SEED + MODELS.index(model)*100)
        cap_col = f'{model}_caption'
        bleu_col = f'bleu4_{model}'

        a1_list, a2_list, a3_list = [], [], []
        hc_list, et_list, etok_list = [], [], []

        for _, row in df.iterrows():
            scenario   = row.get('scenario_category','general_baseline')
            base_rate  = SCENARIO_ERR_RATE.get(scenario, 0.25)
            bleu       = row.get(bleu_col, 0.2) or 0.2
            # Lower BLEU → higher error probability (imperfect proxy)
            err_prob   = min(0.9, max(0.05, base_rate * (1 + (0.25 - bleu) * 1.5)))
            is_error   = rng.rand() < err_prob

            # Annotator labels with ~15% disagreement
            def ann_label(base):
                return (not base) if rng.rand() < 0.15 else base
            a1, a2, a3 = ann_label(is_error), ann_label(is_error), ann_label(is_error)
            majority   = Counter([a1, a2, a3]).most_common(1)[0][0]

            if majority:
                probs = MODEL_ERR_DIST.get(model, [0.25]*4)
                et    = rng.choice(ERROR_TYPES, p=probs)
                cap   = str(row.get(cap_col,''))
                toks  = [t for t in cap.split()
                         if t.lower() not in {'a','an','the','in','on','at','of','and'}]
                etok  = rng.choice(toks) if toks else '[UNK]'
            else:
                et, etok = 'correct', None

            a1_list.append(int(not a1)); a2_list.append(int(not a2)); a3_list.append(int(not a3))
            hc_list.append(int(not majority))
            et_list.append(et); etok_list.append(etok)

        df[f'annotator_1_{model}'] = a1_list
        df[f'annotator_2_{model}'] = a2_list
        df[f'annotator_3_{model}'] = a3_list
        df[f'human_correct_{model}'] = hc_list
        df[f'error_type_{model}']    = et_list
        df[f'error_token_{model}']   = etok_list

    print('✅ Simulation complete.')
    print('Annotation summary (error rates):')
    for model in MODELS:
        n_err = (df[f'human_correct_{model}']==0).sum()
        print(f'  {cfg.MODEL_DISPLAY.get(model, model):<12}: '
              f'{n_err}/{len(df)} errors ({n_err/len(df)*100:.1f}%)')

## 📊 Section 2: Inter-Annotator Agreement

In [ ]:
# CELL 7 — Cohen's Kappa + Krippendorff's Alpha
def krippendorff_alpha(data):
    n_ann, n_items = data.shape
    Do = sum(1 for item in range(n_items)
             for i in range(n_ann) for j in range(i+1, n_ann)
             if data[i,item] != data[j,item])
    n_pairs = n_items * n_ann * (n_ann-1) / 2
    Do /= max(n_pairs, 1)
    vals = data.flatten()
    u, c = np.unique(vals, return_counts=True)
    De = 1 - sum((ci/len(vals))**2 for ci in c)
    return 1 - Do/De if De > 0 else 1.0

print(f'{"Model":<12} {"κ(1,2)":>8} {"κ(1,3)":>8} {"κ(2,3)":>8} {"α(Kripp)":>10}')
print('-'*52)
iaa_rows = []
for model in MODELS:
    a1 = df[f'annotator_1_{model}'].values.astype(int)
    a2 = df[f'annotator_2_{model}'].values.astype(int)
    a3 = df[f'annotator_3_{model}'].values.astype(int)
    k12 = cohen_kappa_score(a1,a2)
    k13 = cohen_kappa_score(a1,a3)
    k23 = cohen_kappa_score(a2,a3)
    alpha = krippendorff_alpha(np.array([a1,a2,a3]))
    print(f'{cfg.MODEL_DISPLAY.get(model,model):<12} {k12:>8.3f} {k13:>8.3f} {k23:>8.3f} {alpha:>10.3f}')
    iaa_rows.append({'model':model,'kappa_1_2':k12,'kappa_1_3':k13,
                     'kappa_2_3':k23,'krippendorff_alpha':alpha})

pd.DataFrame(iaa_rows).to_csv(OUTPUT_DIR/'iaa_scores.csv', index=False)
print('\n💾 iaa_scores.csv saved')
print('κ > 0.61 = substantial | κ > 0.80 = almost perfect')

In [ ]:
# CELL 8 — Error type distribution
error_palette = {k:v['color'] for k,v in cfg.ERROR_TAXONOMY.items()}
fig, axes = plt.subplots(1, len(MODELS), figsize=(4*len(MODELS), 5))
fig.suptitle('Error Type Distribution per Model', fontsize=13, fontweight='bold')

for ax, model in zip(axes, MODELS):
    col = f'error_type_{model}'
    if col not in df.columns: continue
    counts = df[col].value_counts()
    colors = [error_palette.get(k,'#95a5a6') for k in counts.index]
    bars = ax.bar(counts.index, counts.values, color=colors, edgecolor='white')
    for bar,cnt in zip(bars,counts.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                str(cnt), ha='center', fontsize=8)
    ax.set_title(cfg.MODEL_DISPLAY.get(model,model), fontweight='bold')
    ax.tick_params(axis='x', rotation=40)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig.savefig(FIGURES_DIR/'error_type_distribution.png', dpi=150, bbox_inches='tight')
plt.show(); print('📊 Saved.')

## ⚡ Section 3: Metric–Human Misalignment

In [ ]:
# CELL 9 — Misalignment computation
BLEU_THRESHOLD = 0.20
print(f'Metric threshold: BLEU-4 ≥ {BLEU_THRESHOLD} → metric says "correct"\n')

misalign_rows = []
for model in MODELS:
    bc  = f'bleu4_{model}'
    hc  = f'human_correct_{model}'
    if bc not in df.columns or hc not in df.columns: continue

    mc  = (df[bc] >= BLEU_THRESHOLD).astype(int)
    hcv = df[hc].astype(int)
    n   = len(df)

    fh  = ((mc==1)&(hcv==0)).sum()   # False High
    fl  = ((mc==0)&(hcv==1)).sum()   # False Low
    ag  = (mc==hcv).sum()            # Agreement

    df[f'metric_correct_{model}'] = mc

    print(f'{cfg.MODEL_DISPLAY.get(model,model)}')
    print(f'  False High (metric=OK, human=FAIL): {fh:3d} ({fh/n*100:.1f}%)')
    print(f'  False Low  (metric=FAIL, human=OK): {fl:3d} ({fl/n*100:.1f}%)')
    print(f'  Agreement:                          {ag:3d} ({ag/n*100:.1f}%)\n')

    misalign_rows.append({
        'model':cfg.MODEL_DISPLAY.get(model,model), 'n_total':n,
        'false_high':fh,'false_high_pct':fh/n*100,
        'false_low':fl,'false_low_pct':fl/n*100,
        'misalign_pct':(fh+fl)/n*100,'agreement_pct':ag/n*100
    })

misalign_df = pd.DataFrame(misalign_rows)
misalign_df.to_csv(OUTPUT_DIR/'misalignment_summary.csv', index=False)
print('💾 misalignment_summary.csv saved')

In [ ]:
# CELL 10 — Confusion matrix: metric vs human
fig, axes = plt.subplots(1, len(MODELS), figsize=(5*len(MODELS), 5))
fig.suptitle('Confusion Matrix: Metric Verdict vs Human Verdict',
             fontsize=13, fontweight='bold')

for ax, model in zip(axes, MODELS):
    mc_col = f'metric_correct_{model}'
    hc_col = f'human_correct_{model}'
    if mc_col not in df.columns: continue

    cm  = confusion_matrix(df[hc_col].astype(int), df[mc_col].astype(int))
    pct = cm / cm.sum() * 100
    ann = np.array([[f'{cm[i,j]}\n({pct[i,j]:.1f}%)' for j in range(2)] for i in range(2)])

    sns.heatmap(cm, ax=ax, annot=ann, fmt='', cmap='Blues', square=True,
                xticklabels=['Metric:0','Metric:1'],
                yticklabels=['Human:0','Human:1'],
                linewidths=0.5, cbar=False)
    ax.set_title(cfg.MODEL_DISPLAY.get(model,model), fontweight='bold')

plt.tight_layout()
fig.savefig(FIGURES_DIR/'confusion_matrix_metric_vs_human.png', dpi=150, bbox_inches='tight')
plt.show(); print('📊 Saved.')

In [ ]:
# CELL 11 — Misalignment bar chart
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(misalign_df)); w = 0.35
b1 = ax.bar(x-w/2, misalign_df['false_high_pct'], w,
            label='False High (metric=OK, human=FAIL)', color='#e74c3c', alpha=0.85)
b2 = ax.bar(x+w/2, misalign_df['false_low_pct'],  w,
            label='False Low (metric=FAIL, human=OK)', color='#3498db', alpha=0.85)
for bars in [b1, b2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+0.3, f'{h:.1f}%', ha='center', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(misalign_df['model'], fontsize=11)
ax.set_ylabel('Images (%)'); ax.set_title('Metric–Human Misalignment per Model',
    fontsize=13, fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(FIGURES_DIR/'misalignment_bar.png', dpi=150, bbox_inches='tight')
plt.show(); print('📊 Saved.')

In [ ]:
# CELL 12 — Metric as classifier: precision/recall report
print('Metric as Classifier — Classification Report (BLEU-4 threshold = 0.20)\n')
for model in MODELS:
    mc_col = f'metric_correct_{model}'
    hc_col = f'human_correct_{model}'
    if mc_col not in df.columns: continue
    print(f'── {cfg.MODEL_DISPLAY.get(model,model)} ──')
    print(classification_report(
        df[hc_col].astype(int), df[mc_col].astype(int),
        target_names=['Incorrect(0)','Correct(1)'], zero_division=0
    ))

In [ ]:
# CELL 13 — Qualitative examples of misalignment
model = MODELS[0]   # BLIP
mc_col = f'metric_correct_{model}'
hc_col = f'human_correct_{model}'
cap_col = f'{model}_caption'
et_col  = f'error_type_{model}'

fh_cases = df[(df[mc_col]==1)&(df[hc_col]==0)].head(4)
fl_cases = df[(df[mc_col]==0)&(df[hc_col]==1)].head(3)

print(f'❌ FALSE HIGH CASES (metric=correct, human=FAIL) — {cfg.MODEL_DISPLAY[model]}')
print('   Model looks good on paper but is actually wrong.')
for _, row in fh_cases.iterrows():
    print(f'\n  id={row["image_id"]}  [{row.get("scenario_category","")}]')
    print(f'  GT:      {str(row["primary_gt_caption"])[:85]}')
    print(f'  Caption: {str(row.get(cap_col,"N/A"))[:85]}')
    print(f'  BLEU-4:  {row.get(f"bleu4_{model}","N/A"):.3f}  Error: {row.get(et_col,"?")}' )

print(f'\n\n⚠️  FALSE LOW CASES (metric=FAIL, human=correct) — {cfg.MODEL_DISPLAY[model]}')
print('   Metric penalizes a valid paraphrase.')
for _, row in fl_cases.iterrows():
    print(f'\n  id={row["image_id"]}  [{row.get("scenario_category","")}]')
    print(f'  GT:      {str(row["primary_gt_caption"])[:85]}')
    print(f'  Caption: {str(row.get(cap_col,"N/A"))[:85]}')
    print(f'  BLEU-4:  {row.get(f"bleu4_{model}","N/A"):.3f}  Human: CORRECT')

In [ ]:
# CELL 14 — Save annotated DataFrame
out_path = OUTPUT_DIR / 'annotated_all_models.csv'
df.to_csv(out_path, index=False)
ann_cols = [c for c in df.columns if any(
    c.startswith(p) for p in
    ['annotator_','human_correct_','error_type_','error_token_','metric_correct_']
)]
print(f'💾 annotated_all_models.csv saved: {out_path}')
print(f'   Annotation columns: {len(ann_cols)}')
print('\n✅ Notebook 4 COMPLETE — run 05_attention_analysis.ipynb next')